# 05 — Semi-supervised learning: label-budget experiments

**Purpose.** Measure how Label Spreading and Label Propagation behave when only a fraction of training labels is
available, compared with a supervised random forest trained on the **same labelled subset** (Proposal RQ1, SRS 1.4).

**Protocol (ML-06).**
* A stratified **pool** is drawn from the training split (`CS_SEMI_POOL_SIZE`). For each label budget and seed, labels
  outside the budget are replaced by `-1` *before* fitting; the true labels of those rows are used only afterwards to
  report transductive accuracy.
* The graph's neighbourhood size is tuned on a **validation** sample.
* Results are reported on a stratified sample of the **test** split, which is never used for any choice.
* Budgets are applied per class with at least one labelled example per class.

Graph methods scale poorly: the `rbf` kernel needs an n×n dense matrix (20 000 rows ≈ 3.2 GB), so the sparse `knn`
kernel and a bounded pool are used.

In [ ]:
# --- Project bootstrap: make backend/app/ml and notebooks/nb_utils importable ---------------------
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not ((_root / "backend" / "app" / "ml").is_dir() and (_root / "notebooks").is_dir()):
    if _root.parent == _root:
        raise RuntimeError("Start Jupyter inside the CyberSentinel repository (for example from notebooks/).")
    _root = _root.parent
for _path in (str(_root / "backend"), str(_root / "notebooks")):
    if _path not in sys.path:
        sys.path.insert(0, _path)

from nb_utils import load_config

CFG = load_config()  # all settings come from CS_* environment variables (see nb_utils/config.py)
CFG.describe()

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
from sklearn.semi_supervised import LabelPropagation, LabelSpreading

from app.ml.constants import BENIGN_LABEL
from app.ml.evaluation import metrics as M
from nb_utils import io, models, plotting, splits
from nb_utils.experiments import record_experiment, save_json
from nb_utils.semi_supervised import UNLABELED, make_label_mask

warnings.filterwarnings("ignore")
STAGE = CFG.staging_dir / "semi_supervised"
STAGE.mkdir(parents=True, exist_ok=True)
FAST = CFG.fast_mode

In [ ]:
splits_ = io.load_splits(CFG.processed_dir)
prep_report = io.load_json(CFG.processed_dir / "preprocessing_report.json")
preprocessor = io.load_object(CFG.processed_dir / "preprocessor.joblib")
FEATURES = [f["name"] for f in prep_report["input_features"]]

pool = splits.stratified_sample(splits_["train"], "label_category", CFG.semi_pool_size, CFG.seed)
CLASSES = models.class_order(pool["label_category"])
tune = splits.stratified_sample(splits_["validation"], "label_category", min(CFG.semi_eval_size, 10_000), CFG.seed)
evaluation = splits.stratified_sample(splits_["test"], "label_category", CFG.semi_eval_size, CFG.seed)
excluded = int((~evaluation["label_category"].isin(CLASSES)).sum())
tune = tune[tune["label_category"].isin(CLASSES)]
evaluation = evaluation[evaluation["label_category"].isin(CLASSES)]

X_pool, y_pool = preprocessor.transform(pool[FEATURES]), models.encode(pool["label_category"], CLASSES)
X_tune, y_tune = preprocessor.transform(tune[FEATURES]), models.encode(tune["label_category"], CLASSES)
X_eval, y_eval = preprocessor.transform(evaluation[FEATURES]), models.encode(evaluation["label_category"], CLASSES)
print(f"pool {len(pool)} | tuning (validation) {len(tune)} | evaluation (test) {len(evaluation)} "
      f"| evaluation rows of classes absent from the pool: {excluded}")

In [ ]:
def score(model, X, y_true):
    labels_true = np.asarray(CLASSES, dtype=object)[y_true]
    labels_pred = np.asarray(CLASSES, dtype=object)[model.predict(X)]
    report = M.classification_report_dict(labels_true, labels_pred, CLASSES)
    return {"macro_f1": report["macro"]["f1"], "weighted_f1": report["weighted"]["f1"], "accuracy": report["accuracy"],
            "false_alarm_rate": M.false_alarm_rate(labels_true, labels_pred, BENIGN_LABEL)}

## 1. Neighbourhood size (validation sample, 10 % labels)

In [ ]:
K_GRID = [5, 10] if FAST else [5, 10, 20]
y_masked, _ = make_label_mask(y_pool, 0.1, CFG.seed)
tuning = pd.DataFrame([
    {"n_neighbors": k, **score(LabelSpreading(kernel="knn", n_neighbors=k, alpha=0.2, max_iter=100).fit(X_pool, y_masked), X_tune, y_tune)}
    for k in K_GRID
])
BEST_K = int(tuning.sort_values("macro_f1", ascending=False).iloc[0]["n_neighbors"])
print("selected n_neighbors:", BEST_K)
tuning

## 2. Label-budget experiment

In [ ]:
rows = []
for budget in CFG.label_budgets:
    for seed in CFG.semi_seeds:
        y_masked, labeled = make_label_mask(y_pool, budget, seed)
        assert int((y_masked != UNLABELED).sum()) == int(labeled.sum()), "masked labels must stay hidden"
        common = {"label_budget": budget, "seed": seed, "labeled_samples": int(labeled.sum())}
        for name, estimator in (
            ("label_spreading", LabelSpreading(kernel="knn", n_neighbors=BEST_K, alpha=0.2, max_iter=100)),
            ("label_propagation", LabelPropagation(kernel="knn", n_neighbors=BEST_K, max_iter=1000)),
        ):
            fitted, seconds = models.fit_timed(estimator, X_pool, y_masked)
            hidden = ~labeled
            transductive = float((fitted.transduction_[hidden] == y_pool[hidden]).mean()) if hidden.any() else None
            rows.append({"method": name, **common, "train_seconds": round(seconds, 3), "transductive_accuracy": transductive,
                         **score(fitted, X_eval, y_eval)})
        baseline, seconds = models.fit_timed(models.make_random_forest(seed, CFG.n_jobs, FAST), X_pool[labeled], y_pool[labeled])
        rows.append({"method": "supervised_random_forest", **common, "train_seconds": round(seconds, 3),
                     "transductive_accuracy": None, **score(baseline, X_eval, y_eval)})
results = pd.DataFrame(rows)
results.groupby(["method", "label_budget"])["macro_f1"].agg(["mean", "std", "count"]).round(4)

In [ ]:
paired = results.pivot_table(index=["label_budget", "seed"], columns="method", values="macro_f1")
paired["delta_spreading_vs_supervised"] = paired["label_spreading"] - paired["supervised_random_forest"]
paired["delta_propagation_vs_supervised"] = paired["label_propagation"] - paired["supervised_random_forest"]
delta = paired.groupby(level=0)[["delta_spreading_vs_supervised", "delta_propagation_vs_supervised"]].agg(["mean", "std"]).round(4)
plotting.label_budget_curve(results, "macro_f1", "Macro-F1 vs label budget (test sample)", CFG.figures_dir / "05_label_budget_macro_f1.png")
delta

## 3. Record the experiment

In [ ]:
record_experiment(
    experiment_id=f"{CFG.family}-semi-supervised-label-budget",
    experiment_type="semi_supervised_label_budget",
    title="Label Spreading / Label Propagation vs supervised baseline across label budgets",
    family=CFG.family,
    rows=results,
    columns={"label_budget": "fraction of pool labels visible during training", "macro_f1": "macro-F1 on the test sample",
             "transductive_accuracy": "accuracy on the pool rows whose labels were hidden"},
    config={"pool_size": len(pool), "evaluation_size": len(evaluation), "tuning_size": len(tune), "kernel": "knn",
            "n_neighbors": BEST_K, "alpha": 0.2, "seeds": CFG.semi_seeds, "budgets": CFG.label_budgets, "fast_mode": FAST},
    summary={"mean_delta_by_budget": {str(b): {"spreading": float(v[0]), "propagation": float(v[2])}
                                      for b, v in zip(delta.index, delta.to_numpy())}},
    notes=["Budgets are applied per class with at least one labelled example per class.",
           "Evaluation uses a stratified sample of the test split that is never used for tuning."],
    experiments_dir=CFG.experiments_dir, reports_dir=CFG.reports_dir,
)
print("experiment saved")

## 4. Optional: stage a Label Spreading model for export (`CS_EXPORT_SEMI=1`)
The fitted model stores its training pool (memory and latency grow with the pool), so it is exported only on request, as a separate `semi_supervised` bundle in notebook 06.

In [ ]:
if os.environ.get("CS_EXPORT_SEMI", "0") == "1":
    budget = float(os.environ.get("CS_SEMI_EXPORT_BUDGET", "0.1"))
    y_masked, labeled = make_label_mask(y_pool, budget, CFG.seed)
    final = LabelSpreading(kernel="knn", n_neighbors=BEST_K, alpha=0.2, max_iter=100).fit(X_pool, y_masked)
    io.save_object(final, STAGE / "label_spreading.joblib")
    save_json({"classes": CLASSES, "label_budget": budget, "n_neighbors": BEST_K, "pool_rows": len(pool),
               "labeled_samples": int(labeled.sum()), "test_sample_metrics": score(final, X_eval, y_eval)}, STAGE / "semi_meta.json")
    print("staged Label Spreading model trained with", int(labeled.sum()), "labels")
else:
    print("CS_EXPORT_SEMI is not set — no semi-supervised model staged.")